# Notebook for analysis for of quantity 4

Number and rate of deaths for each county

### Method: 

Each county is analyzed separately. 
Due to expected low counts, Poisson distribution is assumed.

To maintain a general strategy for all counties, the largest mortality crisis identified is assumed to be the pandemic. Hence the estimate for any given county is simply the largest mortality crisis, even if this occured at a time very different from the pandemic period.

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt 

from tqdm import tqdm

# Load main functions
import ExcessMortalityFunctions as emf


pathData = 'data/'
pathFigs = 'figures/'

# Load full data files

In [2]:
df1 = pd.read_csv('data/KY_deaths - v1.csv',parse_dates=[2])
df2 = pd.read_csv('data/MD_deaths - v1.csv',parse_dates=[2])
df1_raw = df1.copy()
df2_raw = df2.copy()

df1.age = df1.age.fillna(-1) # Code missing ages as -1
df2.age = df2.age.fillna(-1) # Code missing ages as -1

In [3]:
dfPop1 = pd.read_csv('data/KY_population - v1.csv')
dfPop2 = pd.read_csv('data/MD_population - v1.csv')

# Construct dataframes of time-series for each county

In [4]:
# Hardcoded period
d1 = np.datetime64('1911-01-01')
d2 = np.datetime64('1930-12-31')
dr = np.arange(d1,d2 + np.timedelta64(1,'D'))
ds = dr[:-1]

In [5]:

dfCounty_KY = df1.groupby(['date','county']).deaths.sum().reset_index().pivot_table(columns='county',index='date',values='deaths')
dfCounty_KY = dfCounty_KY.reindex(dr) # Ensure that all dates are included
dfCounty_KY = dfCounty_KY.fillna(0) # Fill all missing entries with zeros

dfCounty_MD = df2.groupby(['date','county']).deaths.sum().reset_index().pivot_table(columns='county',index='date',values='deaths')
dfCounty_MD = dfCounty_MD.reindex(dr) # Ensure that all dates are included
dfCounty_MD = dfCounty_MD.fillna(0) # Fill all missing entries with zeros

d1_MD = df2.date.min()
d2_MD = df2.date.max()

In [6]:
dfCountyPop_KY = dfPop1.groupby(['year','county'])['population'].sum()
dfCountyPop_KY = dfCountyPop_KY.reset_index().pivot_table(index='year',columns='county',values='population')

dfCountyPop_KY.index = pd.to_datetime(dfCountyPop_KY.index,format='%Y')
dfCountyPop_KY = dfCountyPop_KY.reindex(np.arange(dfCountyPop_KY.index.min(),dfCountyPop_KY.index.max()+np.timedelta64(1,'D'),np.timedelta64(1,'D')))
dfCountyPop_KY = dfCountyPop_KY.interpolate()

# Same with MD
dfCountyPop_MD = dfPop2.groupby(['year','county'])['population'].sum()
dfCountyPop_MD = dfCountyPop_MD.reset_index().pivot_table(index='year',columns='county',values='population')

dfCountyPop_MD.index = pd.to_datetime(dfCountyPop_MD.index,format='%Y')
dfCountyPop_MD = dfCountyPop_MD.reindex(np.arange(dfCountyPop_MD.index.min(),dfCountyPop_MD.index.max()+np.timedelta64(1,'D'),np.timedelta64(1,'D')))
dfCountyPop_MD = dfCountyPop_MD.interpolate()

# Set up helper function for analysis

In [ ]:
numYears = 3 # Baseline based on data from -3 year to +3, i.e. a 6-year baseline
# timeResolution = 'Month' 
timeResolution = 'Week' 
# timeResolution = 'Day' # All anaylsis done on daily level and aggregated when necessary
ThresholdToUse = 3 # Zscore threshold to use for data to be omitted from baseline

In [ ]:

def curAnalysisFunction(curData):
    # Calculate mean and standard deviation
    curBaseline,curStandarDeviation = emf.rnMean(curData,numYears=numYears,timeResolution=timeResolution,distributionType='Standard')


    # Remove all above threshold
    _,curBaselineFull,curStandardDeviationFull = emf.removeAboveThresholdAndRecalculateRepeatFull(curData,ZscoreThreshold=ThresholdToUse,numYears=numYears,timeResolution=timeResolution,distributionType='Standard')


    # Determine full excess mortality from baseline
    curExcFull,curZscFull,curExcPctFull = emf.getExcessAndZscore(curData,curBaselineFull,curStandardDeviationFull)

    curTime = curBaseline.index 
    curExcess = curData - curBaselineFull
    curZscore = curZscFull

    # Identify all mortality crises.
    dateGroups,allExcess = emf.determineMortalityCrisis(curTime,curExcess,curZscore,upperThreshold=4,lowerThreshold=2,maxDaysBelowThreshold=7,minDurationOfCrisis=0,returnExcessCount=True)

    dateGroups[:5],allExcess[:5]

    # Count only mortality crises occuring in the general pandemic period
    allExcessDuringPandemic = 0
    allExcessDuringPandemic_lo = 0
    allExcessDuringPandemic_hi = 0
    for i in range(len(dateGroups)):
        curTotalExcess = allExcess[i]
        curDateGroup = dateGroups[i]
        if curDateGroup[0] >= np.datetime64('1918-03-01'):
            if curDateGroup[1] < np.datetime64('1921-05-01'):
                # print(curDateGroup)
                # allExcessDuringPandemic += curTotalExcess

                dStart = curDateGroup[0]
                dEnd = curDateGroup[1]

                total_observed = curData.loc[dStart:dEnd].sum()
                total_baseline = curBaselineFull.loc[dStart:dEnd].sum()
                total_baseline_lo = (curBaselineFull + 2 * curStandardDeviationFull).loc[dStart:dEnd].sum()
                total_baseline_hi = (curBaselineFull - 2 * curStandardDeviationFull).loc[dStart:dEnd].sum()

                # print(int(total_observed.round()),int(total_baseline.round()),int(total_baseline_lo.round()),int(total_baseline_hi.round()))

                excess = total_observed - total_baseline
                excess_lo = total_observed - total_baseline_lo
                excess_hi = total_observed - total_baseline_hi

                allExcessDuringPandemic += excess
                allExcessDuringPandemic_lo += excess_lo
                allExcessDuringPandemic_hi += excess_hi

    return allExcessDuringPandemic,allExcessDuringPandemic_lo,allExcessDuringPandemic_hi


# curData = dfCounty_KY.iloc[:,0].copy()
# curData = emf.groupByWeek(curData)
# curRes =  curAnalysisFunction(curData)

In [ ]:
dfResults_KY = pd.DataFrame(index=dfCounty_KY.columns,columns=['Excess','ExcessLo','ExcessHi'])

for i,col in enumerate(tqdm(dfCounty_KY.columns)):
    
    curData = dfCounty_KY[col]
    curData = emf.groupByWeek(curData)

    curRes = curAnalysisFunction(curData)

    dfResults_KY.loc[col,'Excess'] = curRes[0]
    dfResults_KY.loc[col,'ExcessLo'] = curRes[1]
    dfResults_KY.loc[col,'ExcessHi'] = curRes[2]

  0%|          | 0/117 [00:00<?, ?it/s]

100%|██████████| 117/117 [00:57<00:00,  2.03it/s]


In [ ]:
dfResults_MD = pd.DataFrame(index=dfCounty_MD.columns,columns=['Excess','ExcessLo','ExcessHi'])

for i,col in enumerate(tqdm(dfCounty_MD.columns)):
    
    curData = dfCounty_MD[col].loc[d1_MD:d2_MD]
    curData = emf.groupByWeek(curData)

    curRes = curAnalysisFunction(curData)

    dfResults_MD.loc[col,'Excess'] = curRes[0]
    dfResults_MD.loc[col,'ExcessLo'] = curRes[1]
    dfResults_MD.loc[col,'ExcessHi'] = curRes[2]

100%|██████████| 21/21 [00:07<00:00,  2.79it/s]


In [ ]:
display(dfResults_KY.head(10))
display(dfResults_MD.head(10))

,Excess,ExcessLo,ExcessHi
county,,,
Adair County,157.433333,21.075996,293.790671
Allen County,77.7,9.533273,145.866727
Anderson County,94.783333,7.254707,182.311959
Ballard County,48.05,-38.002032,134.102032
Barren County,178.983333,73.275267,284.6914
Bath County,102.133333,-26.640634,230.907301
Bell County,170.866667,113.92698,227.806353
Boone County,35.266667,-46.836245,117.369578
Bourbon County,122.2,-78.934362,323.334362


,Excess,ExcessLo,ExcessHi
county,,,
Allegany,800.033333,665.612096,934.454571
Calvert,89.983333,-32.752666,217.385999
Caroline,146.533333,19.636527,273.43014
Carroll,244.25,64.206019,424.293981
Cecil,174.233333,102.025689,246.440977
Charles,165.683333,65.643489,265.723177
Dorchester,240.55,72.641585,408.458415
Frederick,407.1,145.875238,705.324762
Garrett,120.1,-60.181433,308.381433


## Calculate by population

In [ ]:

dfResultsPerPop_KY = pd.DataFrame(index=dfCounty_KY.columns,columns=['Excess','ExcessLo','ExcessHi'])
dfResultsPerPop_MD = pd.DataFrame(index=dfCounty_MD.columns,columns=['Excess','ExcessLo','ExcessHi'])


In [ ]:
# Simply use the population estimate at the earliest point considered
curPops_KY = dfCountyPop_KY.loc['1918-03-01']
curPops_MD = dfCountyPop_MD.loc['1918-03-01']

# Rename misnamed county
curPops_KY = curPops_KY.rename(index={'Larue County': 'LaRue County'})

In [ ]:
for i,col in enumerate(tqdm(dfCounty_KY.columns)):
    curPop = curPops_KY[col]
    curExc = dfResults_KY.loc[col,'Excess']
    curExc_lo = dfResults_KY.loc[col,'ExcessLo']
    curExc_hi = dfResults_KY.loc[col,'ExcessHi']

    
    dfResultsPerPop_KY.loc[col,'Excess'] = curExc / curPop
    dfResultsPerPop_KY.loc[col,'ExcessLo'] = curExc_lo / curPop
    dfResultsPerPop_KY.loc[col,'ExcessHi'] = curExc_hi / curPop

100%|██████████| 117/117 [00:00<00:00, 6924.13it/s]


In [ ]:

for i,col in enumerate(tqdm(dfCounty_MD.columns)):
    curPop = curPops_MD[col+ ' County']
    # print(col,curPop)
    curExc = dfResults_MD.loc[col,'Excess']
    curExc_lo = dfResults_MD.loc[col,'ExcessLo']
    curExc_hi = dfResults_MD.loc[col,'ExcessHi']

    
    dfResultsPerPop_MD.loc[col,'Excess'] = curExc / curPop
    dfResultsPerPop_MD.loc[col,'ExcessLo'] = curExc_lo / curPop
    dfResultsPerPop_MD.loc[col,'ExcessHi'] = curExc_hi / curPop
# dfResultsPerPop_MD.head()
# # dfResults_MD.head()

100%|██████████| 21/21 [00:00<00:00, 5249.75it/s]


In [ ]:
dfResults_KY.to_excel('Quantity4_County_KY.xlsx')
dfResults_MD.to_excel('Quantity4_County_MD.xlsx')
dfResultsPerPop_KY.to_excel('Quantity4_County_Rate_KY.xlsx')
dfResultsPerPop_MD.to_excel('Quantity4_County_Rate_MD.xlsx')